In [1]:
import sys
from pathlib import Path
import os

# Add the project root to Python path so imports work from subdirectories
# Try multiple methods to find the project root
current_dir = Path(os.getcwd())

# Method 1: Go up from current directory (assumes we're in VarianceDecisionTree/process_data)
project_root = current_dir.parent.parent

# Method 2: If that doesn't have main.py, try to find it by looking for main.py
if not (project_root / "main.py").exists():
    # Walk up until we find main.py
    test_path = current_dir
    while test_path != test_path.parent:
        if (test_path / "main.py").exists():
            project_root = test_path
            break
        test_path = test_path.parent

project_root_str = str(project_root.resolve())

# Remove project_root from path if it's there, then add it at the beginning
if project_root_str in sys.path:
    sys.path.remove(project_root_str)
sys.path.insert(0, project_root_str)

# Import utils from the project root (not the installed package)
# Force reload if already imported to avoid conflicts
if 'utils' in sys.modules:
    # Check if it's the wrong utils (from site-packages)
    utils_module = sys.modules['utils']
    utils_file = getattr(utils_module, '__file__', None)
    if utils_file and 'site-packages' in str(utils_file):
        del sys.modules['utils']
        # Also remove any submodules
        keys_to_remove = [k for k in sys.modules.keys() if k.startswith('utils.')]
        for k in keys_to_remove:
            del sys.modules[k]

import utils
# Verify we imported the correct utils module (should have 'announce' and 'unzip' functions)
if not hasattr(utils, 'announce'):
    utils_file = getattr(utils, '__file__', 'unknown location')
    raise ImportError(f"Wrong utils module imported! Expected local utils.py with 'announce' function. "
                     f"Got: {utils_file}")
if not hasattr(utils, 'unzip'):
    utils_file = getattr(utils, '__file__', 'unknown location')
    raise ImportError(f"Wrong utils module imported! Expected local utils.py with 'unzip' function. "
                     f"Got: {utils_file}")

from BenchmarkProblems.GraphColouring import GraphColouring, GraphColouringPrettifier
from config import paths

resources_dir = Path(paths.resources_dir)
problem_file_name = resources_dir / "problem_definitions" / "GC" / "jean.json"
dat_json_file_name = resources_dir / "problem_definitions" / "GC" / "jean.dat.json"
dat_file_name = resources_dir / "problem_definitions" / "GC" / "jean.dat"

original_problem = GraphColouring.from_json(str(problem_file_name))
problem = original_problem  # Will be modified later

# Load GraphColouringPrettifier - use .dat.json if it exists, otherwise load from .dat and save as .dat.json
if dat_json_file_name.exists():
    gcp = GraphColouringPrettifier.from_json(str(dat_json_file_name))
else:
    # Load from .dat file and save as .dat.json for future use
    if dat_file_name.exists():
        gcp = GraphColouringPrettifier.from_dat_file(str(dat_file_name))
        gcp.store_as_json(str(dat_json_file_name))
    else:
        raise FileNotFoundError(f"Neither {dat_json_file_name} nor {dat_file_name} found!")

actual_amount_of_nodes = 30
nodes_by_count = {node_number: len([pair for pair in problem.connections if node_number in pair])
                  for node_number in range(problem.amount_of_nodes)}
jv_index = gcp.abbreviation_list.index("JV")
print(f"The index of JV is {jv_index}, {nodes_by_count[jv_index]}")
nodes_by_count = list(nodes_by_count.items())
nodes_by_count.sort(key=utils.second, reverse=True)
for n, count in nodes_by_count:
    print(n, count, gcp.get_abbreviation_from_node_number(n))
nodes_to_keep_original = [n for n, _ in nodes_by_count[:actual_amount_of_nodes]]
nodes_to_keep = set(nodes_to_keep_original)
# Create mapping from original node indices to new indices (0 to actual_amount_of_nodes-1)
node_mapping = {old_node: new_node for new_node, old_node in enumerate(nodes_to_keep_original)}
problem = GraphColouring(amount_of_nodes=actual_amount_of_nodes,
                         amount_of_colours=3,
                         connections=[(node_mapping[a], node_mapping[b]) for (a, b) in problem.connections 
                                        if a in nodes_to_keep and b in nodes_to_keep])


print(", ".join(map(gcp.get_abbreviation_from_node_number, nodes_to_keep)))

The index of JV is 45, 8
36 72 GU
71 44 SS
14 38 CL
27 34 FT
56 32 MN
2 30 BB
8 30 BS
33 26 GP
65 26 PL
6 24 BO
18 24 CR
37 22 HL
43 22 JP
47 22 LL
58 22 MP
67 22 QU
78 22 XB
3 20 BJ
13 20 CH
34 20 GR
54 20 MI
57 20 MO
5 18 BM
32 18 GI
49 18 MA
4 16 BL
15 14 CM
24 14 FE
26 14 FN
28 14 FV
39 14 JA
46 14 LI
62 14 MY
68 14 SC
72 14 TG
76 14 VI
9 12 BT
16 12 CN
19 12 CV
21 12 EN
38 12 IS
25 8 FF
42 8 JO
45 8 JV
1 6 BA
17 6 CO
29 6 GA
63 6 NP
74 6 TM
75 6 TS
7 4 BR
10 4 BU
22 4 EP
30 4 GE
41 4 JL
44 4 JU
53 4 MG
59 4 MR
73 4 TH
77 4 XA
0 2 AZ
11 2 BZ
12 2 CC
23 2 FA
31 2 GG
35 2 GT
40 2 JD
50 2 MB
51 2 MC
52 2 ME
55 2 MM
60 2 MT
61 2 MV
64 2 PG
66 2 PO
69 2 SN
79 2 ZE
20 0 DA
48 0 LP
70 0 SP
BB, BJ, BL, BM, BO, BS, CH, CL, CM, CR, FE, FN, FT, FV, GI, GP, GR, GU, HL, JP, LL, MA, MI, MN, MO, MP, PL, QU, SS, XB


In [2]:
# Ensure utils is correctly imported before other imports
import sys
if 'utils' in sys.modules:
    utils_module = sys.modules['utils']
    utils_file = getattr(utils_module, '__file__', None)
    if utils_file and 'site-packages' in str(utils_file):
        del sys.modules['utils']
        keys_to_remove = [k for k in sys.modules.keys() if k.startswith('utils.')]
        for k in keys_to_remove:
            del sys.modules[k]
        import utils
        # Verify correct utils
        if not hasattr(utils, 'unzip') or not hasattr(utils, 'announce'):
            raise ImportError(f"Wrong utils module! Missing required functions.")
    else:
        import utils
else:
    import utils

# Verify utils has required functions
if not hasattr(utils, 'unzip'):
    raise ImportError(f"utils module missing 'unzip' function. Got: {getattr(utils, '__file__', 'unknown')}")

from utils import announce
from VarianceDecisionTree.PSDecisionTree import PSDecisionTree
from Explanation.PRefManager import PRefManager

algorithm_suite = [
    "BBO"
]
print(f"Using algorithms: {', '.join(algorithm_suite)}")

with announce("generating the pRef"):
    pRef = PRefManager.generate_pRef(problem=problem,
                                     sample_size=10000,
                                     which_algorithm=" ".join(algorithm_suite))


metrics = "simplicity atomicity"
decision_tree = PSDecisionTree(maximum_depth=3,
                               ps_budget=5000,
                               ps_search_population_size=100,
                               problem=problem,
                               metrics_to_use=metrics)
with announce("training the decision tree"):
    decision_tree.train_from_pRef(pRef, verbose=True)
    

#decision_tree.set_repr_ps(gcp.repr_ps)

print(decision_tree.repr_long())

Using algorithms: BBO
generating the pRef...Running MEALPY BBO on GraphColouring
BBO parameters: pop_size=50
   Available history attributes: ['_History__set_keyword_arguments', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'epoch', 'get_global_repeated_times', 'list_current_best', 'list_current_best_fit', 'list_current_worst', 'list_diversity', 'list_epoch_time', 'list_exploitation', 'list_exploration', 'list_global_best', 'list_global_best_fit', 'list_global_worst', 'list_population', 'log_file', 'log_to', 'logger', 'save_diversity_chart', 'save_exploration_exploitation_chart', 'save_global_best_fitness_chart', 'save_global_objectives_chart', 'save_local_best_fitness_chart', 

TypeError: 'numpy.float64' object is not iterable

In [ ]:
from Core.PS import PS


def repr_factors(ps:PS) -> str:
    fixed_positions = ps.get_fixed_variable_positions()
    return ", ".join(f"{i}-{ps.values[i]}" for i in fixed_positions)

# Create reverse mapping from new indices to original indices
reverse_node_mapping = {new_node: old_node for old_node, new_node in node_mapping.items()}

def repr_ps_with_original_indices(ps: PS) -> str:
    """Wrapper to convert PS with new node indices back to original indices for gcp"""
    # Create a PS with the original problem's search space
    original_ps = PS.empty(original_problem.search_space)
    
    # Map fixed positions from new indices to original indices
    for new_pos in ps.get_fixed_variable_positions():
        if new_pos in reverse_node_mapping:
            original_pos = reverse_node_mapping[new_pos]
            original_ps = original_ps.with_fixed_value(original_pos, ps.values[new_pos])
    
    return gcp.repr_ps(original_ps)

decision_tree.set_repr_ps(repr_ps_with_original_indices)
#decision_tree.set_repr_ps(repr_factors)

NameError: name 'decision_tree' is not defined

In [ ]:
print(decision_tree.repr_long())

NameError: name 'decision_tree' is not defined